# 01 · Per-sample QC metrics and doublet scores

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`preprocess.counts_to_adata`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/preprocess/preprocess.wdl) |
| Script | [`prep_metadata`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/prep_metadata) → `sc-prep-metadata` |

This is the last per-sample step of preprocessing. Its outputs are the starting point of the cohort analysis, and they are already in the curated bucket, so this notebook **inspects** them. Re-running the step on one sample is optional (section 3).

For each sample, `prep_metadata`:

1. reads the CellBender `.h5` into AnnData (`anndata_from_h5`, keeping only barcodes CellBender called as cells),
2. flags mitochondrial (`MT-*`) and ribosomal (`RPL*`, `RPS*`, `MRPL*`, `MRPS*`) genes in `var["mt"]` / `var["rb"]`,
3. computes per-cell QC metrics with `sc.pp.calculate_qc_metrics`: `n_genes_by_counts`, `total_counts`, `pct_counts_mt`, `pct_counts_rb`, …,
4. scores doublets per sample with `sc.pp.scrublet` and keeps `doublet_score` (dropping the binary `predicted_doublet`),
5. adds sample metadata: `sample`, `batch`, `sex`, `brain_region_level_1/2/3`, `team`, `dataset` and `batch_id = f"{team}_{dataset}_{batch}"`, the integration batch key.

Doublets must be scored **per sample** (before merging), because scrublet simulates doublets from cells captured together.

**Inputs**
- `${dataset_id}.${sample_id}.cellbender.h5`: CellBender-cleaned counts (one per sample)

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${dataset_id}.${sample_id}.cleaned_unfiltered.h5ad`

**Previous:** [00_setup](00_setup.ipynb) · **Next:** [02_merge_and_plot_qc](02_merge_and_plot_qc.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. The per-sample files

In [ ]:
import anndata as ad

cleaned = sorted(cfg.PREPROCESS_PATH.glob("*.cleaned_unfiltered.h5ad"))
example = ad.read_h5ad(cleaned[0], backed="r")
print(cleaned[0].name)
example

In [ ]:
print("obs columns:", list(example.obs.columns))
print(f"mitochondrial genes: {int(example.var['mt'].sum())}, ribosomal genes: {int(example.var['rb'].sum())}")
example.obs.head()

## 2. QC summary per sample

Only `obs` is read from each file (backed mode), so this is cheap even for large samples.

In [ ]:
metrics = ["total_counts", "n_genes_by_counts", "pct_counts_mt", "doublet_score"]
obs = []
for path in cleaned:
    a = ad.read_h5ad(path, backed="r")
    obs.append(a.obs[["sample", "batch_id"] + metrics].copy())
    a.file.close()
obs = pd.concat(obs)

summary = obs.groupby("sample", observed=True).agg(
    n_cells=("total_counts", "size"),
    batch_id=("batch_id", "first"),
    median_total_counts=("total_counts", "median"),
    median_n_genes=("n_genes_by_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),
    median_doublet_score=("doublet_score", "median"),
)
print(f"{len(obs):,} cells in {len(summary)} samples")
summary

## 3. Optional: re-run `prep_metadata` on one sample

Set `RERUN = True` to rebuild the first sample from its CellBender output and check it against the curated file. The arguments are read back from the curated `obs`. scrublet uses a fixed random seed, so `doublet_score` should match.

In [ ]:
RERUN = False

if RERUN:
    first = example.obs.iloc[0]
    stem = cleaned[0].name.removesuffix(".cleaned_unfiltered.h5ad")  # ${dataset_id}.${sample_id}
    rerun_path = cfg.RUN_DIR / f"{stem}.cleaned_unfiltered.h5ad"
    cfg.run(
        "sc-prep-metadata",
        "--adata-input", cfg.PREPROCESS_PATH / f"{stem}.cellbender.h5",
        "--sample-id", first["sample"],
        "--batch", first["batch"],
        "--sex", first["sex"],
        "--brain-region-level-1", first.get("brain_region_level_1", "NA"),
        "--brain-region-level-2", first.get("brain_region_level_2", "NA"),
        "--brain-region-level-3", first.get("brain_region_level_3", "NA"),
        "--team", first["team"],
        "--dataset", first["dataset"],
        "--adata-output", rerun_path,
    )
    rerun = ad.read_h5ad(rerun_path, backed="r")
    shared = rerun.obs_names.intersection(example.obs_names)
    print(f"cells: re-run {rerun.n_obs:,}, curated {example.n_obs:,}, shared {len(shared):,}")
    for m in metrics:
        diff = (rerun.obs.loc[shared, m] - example.obs.loc[shared, m]).abs().max()
        print(f"max |difference| in {m}: {diff:.3g}")

**Previous:** [00_setup](00_setup.ipynb) · **Next:** [02_merge_and_plot_qc](02_merge_and_plot_qc.ipynb)